In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
import pycountry

from emu_renewal.inputs import ANALYSIS_TYPES, DATA_PATH, BASE_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_post_prior_comparison, plot_imm_props

In [ ]:
run_name = "42719116"

In [ ]:
run_path = BASE_PATH / "outputs" / run_name

In [ ]:
from os import listdir as ls

countries = ls(run_path)

In [ ]:
gmob_columns = [
    'retail_and_recreation',
    'grocery_and_pharmacy',
    'parks',
    'transit_stations',
    'workplaces',
    'residential'
]

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from itertools import product

def mw_plot(country, w=True, chain=0, ax=None,legend=True):
    c_path = run_path / country
    a_path = c_path/"gmob"        
    idata = az.from_netcdf(a_path / "idata_filtered.nc")
    
    if ax is None:
        fig = plt.figure();
        ax = fig.gca()
    if w:
        #plt.figure();
        df = idata.posterior["mob_weights"].sel(chain=chain).to_pandas()
        df.columns = gmob_columns
        sns.kdeplot(df,ax=ax,legend=legend)
        ax.set_title(f"{country}: mobility weights")
        return ax
    else:
        df = idata.posterior["mob_exp"].sel(chain=chain).to_pandas()
        df.columns = gmob_columns
        ax = sns.kdeplot(df,ax=ax,legend=legend)
        ax.set_title(f"{country}: mobility exponents")
        return ax

In [ ]:
#Plot weights and exponents for single country
country = "ITA"

mw_plot(country,True)
mw_plot(country,False)

In [ ]:
# Plot all weights in grid

fig, axes = plt.subplots(4,4,figsize=(16,16))
for cidx, aidx in enumerate(list(product(range(4),range(4)))):
    if cidx==0:
        legend = True
    else:
        legend = False
    try:
        # True/False switches between weights/exponents
        mw_plot(countries[cidx],True,ax=axes[aidx],legend=legend,chain=2)
    except:
        pass

In [ ]:
# Plot all exponents in grid

fig, axes = plt.subplots(4,4,figsize=(16,16))
for cidx, aidx in enumerate(list(product(range(4),range(4)))):
    if cidx==0:
        legend = True
    else:
        legend = False
    try:
        # True/False switches between weights/exponents
        mw_plot(countries[cidx],False,ax=axes[aidx],legend=legend,chain=3)
    except:
        pass